# CPS intervention planning

**Grand Challenge release notebook.** This self-contained notebook measures or reuses a compact CPS packet and converts its fragile couplings into a preregistered scalar control hypothesis.


## Release contract

- **Scientific question:** Which candidate learning-rate scale minimizes the declared CPS surrogate risk on the measured reduced operator?
- **Default execution path:** Generate a compact Pythia-70M packet internally; `CPS_EVIDENCE_PATH` is an optional reuse path.
- **Evidence boundary:** The planner recommendation is a prospective continuation hypothesis, not a causal result.
- **Primary outputs:** Evidence packet, legible candidate table, `planner_recommendation.json`, and export archive.


## Interpretation checklist

- Confirm whether the packet was generated locally or explicitly reused.
- Inspect all candidate scores rather than only the selected winner.
- Preserve the recommendation before inspecting continuation outcomes.


## Planning contract

The planner minimizes a composite CPS risk over the same selected couplings. Its recommendation is a hypothesis for a matched continuation run. It is not applied to model weights here, and it should not be described as a causal result.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — obtain the evidence packet

The default path is fully self-contained: run a compact Pythia-70M probe and plan against the packet it emits. Set `CPS_EVIDENCE_PATH` only to reuse a deliberate external packet or ZIP archive.


In [ ]:
from cps.notebook import stage_banner
stage_banner('1', 'obtain the evidence packet', objective='The default path is fully selfcontained: run a compact Pythia70M probe and plan against the packet it emits. Set CPSEVIDENCEPATH only to reuse a deliberate external packet or ZIP archive.', deliverable="The artifacts and console evidence described in this stage.")

import os
import numpy as np
from cps.pythia.notebook_support import load_evidence_packet, run_self_contained_probe

evidence_path = os.environ.get("CPS_EVIDENCE_PATH")
if evidence_path:
    print(f"[PLAN] Reusing explicit evidence: {evidence_path}", flush=True)
    packet = load_evidence_packet(evidence_path)
else:
    revision = os.environ.get("CPS_REVISION", "step0")
    print(f"[PLAN] Running a self-contained compact probe at revision={revision}", flush=True)
    generated_root = run_self_contained_probe(
        revision=revision,
        run_name="pythia-70m-planning",
    )
    packet = load_evidence_packet(generated_root)
path = packet.reduced_operator_path
A = packet.matrix
print(f"[PLAN] evidence root={packet.root}", flush=True)
print(f"[PLAN] shape={A.shape}; ||A||₂={np.linalg.norm(A, 2):.6g}", flush=True)


## Stage 2 — score every candidate, not only the winner

The table makes the decision legible. Risk combines phase-envelope spectral radius, finite-horizon gain, and inverse minimum gap. Lower is better under this preregistered surrogate.

In [ ]:
from cps.notebook import stage_banner
stage_banner('2', 'score every candidate, not only the winner', objective='The table makes the decision legible. Risk combines phaseenvelope spectral radius, finitehorizon gain, and inverse minimum gap. Lower is better under this preregistered surrogate.', deliverable="The artifacts and console evidence described in this stage.")

import numpy as np
import pandas as pd
from IPython.display import display
from cps.controllers import score_candidate
from cps.pythia.notebook_support import select_candidate_edges
from cps.pythia.planner import damping_family, plan_scalar_control

edges = select_candidate_edges(packet, maximum=8)
candidate_scales = [1.0, 0.98, 0.95, 0.90, 0.80]
rows = []
for scale in candidate_scales:
    gamma = 1.0 - scale
    score = score_candidate(damping_family(A, gamma), edges)
    rows.append({
        "learning-rate scale": scale,
        "surrogate damping γ": gamma,
        "risk": score.risk,
        "spectral radius": score.spectral_radius,
        "transient gain": score.transient_gain,
        "minimum gap": score.minimum_gap,
    })
score_frame = pd.DataFrame(rows).sort_values("risk")
display(score_frame)
recommendation = plan_scalar_control(
    "learning_rate_scale",
    1.0,
    candidate_scales,
    lambda scale: damping_family(A, 1.0 - scale),
    edges,
)
print("[PLAN] recommendation", recommendation.to_dict(), flush=True)


## Stage 3 — register the hypothesis

The next notebook must compare the recommended control against a baseline from identical weights and identical subsequent batches. Choosing the intervention after seeing continuation outcomes would invalidate the test.

In [ ]:
from cps.notebook import stage_banner
stage_banner('3', 'register the hypothesis', objective='The next notebook must compare the recommended control against a baseline from identical weights and identical subsequent batches. Choosing the intervention after seeing continuation outcomes would invalidate the test.', deliverable="The artifacts and console evidence described in this stage.")

from pathlib import Path
from cps.pythia.notebook_support import write_planner_recommendation

plan_root = Path("/content/cps-artifacts/planning")
plan_root.mkdir(parents=True, exist_ok=True)
plan_path = write_planner_recommendation(
    plan_root,
    recommendation,
    selected_edges=edges,
    candidate_grid=candidate_scales,
    control_operationalization={
        "surrogate_family": "isotropic_damping",
        "translation": "gamma = 1 - learning_rate_scale",
        "warning": "The matched continuation is the actual intervention test.",
    },
)
print(f"[PLAN] preregistered recommendation={plan_path}", flush=True)


## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
